In [1]:
import os
import pandas as pd

try:
    from components.map_view import (
        display_landing_page_map_dots,
        display_landing_page_map_choropleth_counties,
        display_state_level_map,
        display_county_level_map,
        fix_cfips
    )
except ModuleNotFoundError:
    from src.components.map_view import (
        display_landing_page_map_dots,
        display_landing_page_map_choropleth_counties,
        display_state_level_map,
        display_county_level_map,
        fix_cfips
    )

import numpy as np

In [2]:
df = pd.read_csv("../data/processed/smb_enriched.csv",dtype={'cfips_fixed': str, 'cfips': str})  
df['cfips_fixed'] = df['cfips_fixed'].astype(str)
df['cfips'] = df['cfips'].astype(str)
df['cfips_fixed'] = df['cfips_fixed'].apply(fix_cfips)

In [3]:
df.head()

,row_id,cfips,county,state,first_day_of_month,microbusiness_density,active,pct_bb_2017,pct_bb_2018,pct_bb_2019,...,pct_it_workers_2021,median_hh_inc_2017,median_hh_inc_2018,median_hh_inc_2019,median_hh_inc_2020,median_hh_inc_2021,centroid_lat,centroid_lng,state_id,cfips_fixed
0,1001_2019-08-01,1001,Autauga County,Alabama,2019-08-01,3.007682,1249,76.6,78.9,80.6,...,1.1,55317,58786.0,58731,57982.0,62660.0,32.536153,-86.641196,1,01001
1,1001_2019-09-01,1001,Autauga County,Alabama,2019-09-01,2.884870,1198,76.6,78.9,80.6,...,1.1,55317,58786.0,58731,57982.0,62660.0,32.536153,-86.641196,1,01001
2,1001_2019-10-01,1001,Autauga County,Alabama,2019-10-01,3.055843,1269,76.6,78.9,80.6,...,1.1,55317,58786.0,58731,57982.0,62660.0,32.536153,-86.641196,1,01001
3,1001_2019-11-01,1001,Autauga County,Alabama,2019-11-01,2.993233,1243,76.6,78.9,80.6,...,1.1,55317,58786.0,58731,57982.0,62660.0,32.536153,-86.641196,1,01001
4,1001_2019-12-01,1001,Autauga County,Alabama,2019-12-01,2.993233,1243,76.6,78.9,80.6,...,1.1,55317,58786.0,58731,57982.0,62660.0,32.536153,-86.641196,1,01001


In [4]:
latest_year = 2021
latest_date = "2022-10-01"

In [5]:
def calculate_sellability(county_income, sorted_income):
    return round(np.searchsorted(sorted_income, county_income, side="right") / len(sorted_income) * 100, 2)

In [6]:
def calculate_hireability(county_education, sorted_education):
    return round(np.searchsorted(sorted_education, county_education, side="right") / len(sorted_education) * 100, 2)

In [7]:
def calculate_growth_index(county_growth, sorted_growth):
    return round(np.searchsorted(sorted_growth, county_growth, side="right") / len(sorted_growth) * 100, 2)


In [8]:
#sellability index calculation 

sell_df = df[df["first_day_of_month"] == latest_date]
sell_df = sell_df[[f"median_hh_inc_{latest_year}", "cfips", "county", "state"]].dropna()

sorted_income = np.sort(sell_df[f"median_hh_inc_{latest_year}"].values)

sellability_data = []

for cfips in sell_df["cfips"].unique():
    county_data = df[df["cfips"] == cfips].iloc[0]
    county_income = county_data[f"median_hh_inc_{latest_year}"]
    sellability_index = calculate_sellability(county_income, sorted_income)
    
    sellability_data.append({
        "cfips": cfips,
        "county": county_data["county"],
        "state": county_data["state"],
        "sellability_index": sellability_index
    })

sellability_df = pd.DataFrame(sellability_data)

In [9]:
#hireability index calculation 

hire_df = df[df["first_day_of_month"] == latest_date]
hire_df = hire_df[[f"pct_college_{latest_year}", "cfips", "county", "state"]].dropna()

sorted_education = np.sort(hire_df[f"pct_college_{latest_year}"].values)

hireability_data = []

for cfips in hire_df["cfips"].unique():
    county_data = df[df["cfips"] == cfips].iloc[0]
    county_education = county_data[f"pct_college_{latest_year}"]
    hireability_index = calculate_hireability(county_education, sorted_education)
    
    hireability_data.append({
        "cfips": cfips,
        "county": county_data["county"],
        "state": county_data["state"],
        "hireability_index": hireability_index
    })

hireability_df = pd.DataFrame(hireability_data)

In [10]:
#Growth index calculation 

growth_df = df[["cfips", "county", "state", "first_day_of_month", "active"]]
growth_df["first_day_of_month"] = pd.to_datetime(growth_df["first_day_of_month"])
growth_df["year"] = growth_df["first_day_of_month"].dt.year
growth_df["month"] = growth_df["first_day_of_month"].dt.month

growth_df = growth_df[growth_df["month"] == 10]
growth_df = growth_df[["cfips", "county", "state", "active", "year"]]

growth_df = growth_df.pivot(index=["cfips", "county", "state"], columns="year", values="active")
growth_df = growth_df.reset_index()

growth_df.columns = growth_df.columns.astype(str)

growth_df['pct_change_2019_2020'] = (growth_df['2020'] - growth_df['2019']) / growth_df['2019'] * 100
growth_df['pct_change_2020_2021'] = (growth_df['2021'] - growth_df['2020']) / growth_df['2020'] * 100
growth_df['pct_change_2021_2022'] = (growth_df['2022'] - growth_df['2021']) / growth_df['2021'] * 100

growth_df['mean_pct_change'] = growth_df[['pct_change_2019_2020', 'pct_change_2020_2021', 'pct_change_2021_2022']].mean(axis=1)

growth_df = growth_df[["cfips", "county", "state", "mean_pct_change"]].dropna()

sorted_growth = np.sort(growth_df["mean_pct_change"].values)

growth_index_data = []

for _, row in growth_df.iterrows():
    county_growth = row["mean_pct_change"]
    growth_index = calculate_growth_index(county_growth, sorted_growth)

    growth_index_data.append({
        "cfips": row["cfips"],
        "county": row["county"],
        "state": row["state"],
        "growth_index": growth_index
    })

growth_index_df = pd.DataFrame(growth_index_data)

/var/folders/8t/5wsb53gx291_v2jk3nfmym880000gn/T/ipykernel_35779/2789090181.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  growth_df["first_day_of_month"] = pd.to_datetime(growth_df["first_day_of_month"])
/var/folders/8t/5wsb53gx291_v2jk3nfmym880000gn/T/ipykernel_35779/2789090181.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  growth_df["year"] = growth_df["first_day_of_month"].dt.year
/var/folders/8t/5wsb53gx291_v2jk3nfmym880000gn/T/ipykernel_35779/2789090181.py:6: SettingWithCopyWarning: 
A value

In [11]:
growth_index_df.head()

,cfips,county,state,growth_index
0,10001,Kent County,Delaware,92.66
1,10003,New Castle County,Delaware,98.34
2,10005,Sussex County,Delaware,99.55
3,1001,Autauga County,Alabama,73.65
4,1003,Baldwin County,Alabama,82.81


In [12]:
sellability_index_df = sellability_df[["cfips", "sellability_index"]]
hireability_index_df = hireability_df[["cfips", "hireability_index"]]
growth_index_df = growth_index_df[["cfips", "growth_index"]]

In [13]:
df_merged = df.merge(sellability_index_df, on="cfips", how="left").merge(hireability_index_df, on="cfips", how="left").merge(growth_index_df, on="cfips", how="left")

df_merged

,row_id,cfips,county,state,first_day_of_month,microbusiness_density,active,pct_bb_2017,pct_bb_2018,pct_bb_2019,...,median_hh_inc_2019,median_hh_inc_2020,median_hh_inc_2021,centroid_lat,centroid_lng,state_id,cfips_fixed,sellability_index,hireability_index,growth_index
0,1001_2019-08-01,1001,Autauga County,Alabama,2019-08-01,3.007682,1249,76.6,78.9,80.6,...,58731,57982.0,62660.0,32.536153,-86.641196,1,01001,69.24,66.73,73.65
1,1001_2019-09-01,1001,Autauga County,Alabama,2019-09-01,2.884870,1198,76.6,78.9,80.6,...,58731,57982.0,62660.0,32.536153,-86.641196,1,01001,69.24,66.73,73.65
2,1001_2019-10-01,1001,Autauga County,Alabama,2019-10-01,3.055843,1269,76.6,78.9,80.6,...,58731,57982.0,62660.0,32.536153,-86.641196,1,01001,69.24,66.73,73.65
3,1001_2019-11-01,1001,Autauga County,Alabama,2019-11-01,2.993233,1243,76.6,78.9,80.6,...,58731,57982.0,62660.0,32.536153,-86.641196,1,01001,69.24,66.73,73.65
4,1001_2019-12-01,1001,Autauga County,Alabama,2019-12-01,2.993233,1243,76.6,78.9,80.6,...,58731,57982.0,62660.0,32.536153,-86.641196,1,01001,69.24,66.73,73.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122260,56045_2022-06-01,56045,Weston County,Wyoming,2022-06-01,1.803249,101,71.1,73.3,76.8,...,57031,53333.0,65566.0,43.839661,-104.567290,56,56045,76.29,50.94,73.97
122261,56045_2022-07-01,56045,Weston County,Wyoming,2022-07-01,1.803249,101,71.1,73.3,76.8,...,57031,53333.0,65566.0,43.839661,-104.567290,56,56045,76.29,50.94,73.97
122262,56045_2022-08-01,56045,Weston County,Wyoming,2022-08-01,1.785395,100,71.1,73.3,76.8,...,57031,53333.0,65566.0,43.839661,-104.567290,56,56045,76.29,50.94,73.97
122263,56045_2022-09-01,56045,Weston County,Wyoming,2022-09-01,1.785395,100,71.1,73.3,76.8,...,57031,53333.0,65566.0,43.839661,-104.567290,56,56045,76.29,50.94,73.97


In [14]:
file_path = "../data/processed/data_details_smb.csv" 

df_merged.to_csv(file_path, index=False)